# 案例 11 · 发布配置端口写错，SLB 全量 502

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

发布模板把 `listen_port` 写成错误端口，进程正常启动但监听端口与 SLB 后端不一致，健康检查全部失败，全量请求 502。

## 练习目标

- 用“发布时间相关性 + 全量失败”快速决策回滚
- 用 `ss -lntp` 对比配置与监听
- 回滚配置并重新冒烟
- 用 `deploy_gate.py` 做发布门禁与自动回滚

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-11-release")
lab.reset(start_api=True)

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_bad_release()
expected = lab.state["services"]["xhs-note-api"]["port"]
print(f"已注入：服务监听 {expected + 10}，SLB 期望 {expected}")


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**监听端口**

In [ ]:
print(lab.sh("ss -lntp"))

**应用配置**

In [ ]:
print(lab.sh("cat /etc/xhs/note-api.conf"))

**期望端口探活**

In [ ]:
print(lab.curl("http://127.0.0.1:%d/healthz" % lab.state["services"]["xhs-note-api"]["port"]))

**错误端口探活**

In [ ]:
print(lab.curl("http://127.0.0.1:%d/healthz" % (lab.state["services"]["xhs-note-api"]["port"] + 10)))

**发布记录**

In [ ]:
print(json.dumps(lab.state["deploy"], ensure_ascii=False, indent=2))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
svc = lab.state["services"]["xhs-note-api"]
expected = svc["port"]
listening = lab.sh("ss -lntp")
print("期望端口:", expected, "| 是否监听:", f":{expected}" in listening)
print("错误端口:", expected + 10, "| 是否监听:", f":{expected + 10}" in listening)
print("判断：进程存活但期望端口未监听 → 发布配置错误，优先回滚")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
expected = lab.state["services"]["xhs-note-api"]["port"]
# 回滚配置到上一版本端口
lab.write("/etc/xhs/note-api.conf",
          f"listen_host = 127.0.0.1\nlisten_port = {expected}\nworker_processes = 4\n")
lab.restart_api(port=expected)
lab.state["deploy"]["Status"] = "rolled_back"
lab.state["deploy"]["History"].insert(0, {
    "Version": lab.state["deploy"]["CurrentVersion"],
    "Time": time.strftime("%F %H:%M"),
    "Result": "rolled_back",
})
lab.state["deploy"]["CurrentVersion"] = "v2026.11.18-3"
lab._save()
print(lab.curl(f"http://127.0.0.1:{expected}/healthz").splitlines()[-1])


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('期望端口已监听', bool(f":{expected}" in lab.sh("ss -lntp")))
lab.check('健康检查通过', bool('HTTP_CODE=200' in lab.curl(f'http://127.0.0.1:{expected}/healthz')))
lab.check('发布已回滚', bool(lab.state['deploy']['Status'] == 'rolled_back'))

## 步骤 6 · 自动化沉淀

脚本执行端口/健康/冒烟三层门禁，失败时按上一版本配置回滚并重启，回滚后再次冒烟通过才返回成功。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_bad_release()
expected = lab.state["services"]["xhs-note-api"]["port"]
out = lab.auto('deploy_gate.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. 为什么“进程存活”不能作为发布成功判据？
2. 回滚与前滚修复分别适用什么场景？
3. 如何设计灰度发布，把配置错误限制在 10% 流量？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')